In [1]:
import json
import pandas as pd

# Ensure the ADOMD.NET path is in the system path prior to importing Pyadomd
from sys import path
adomd_path = r'C:\Program Files\Microsoft.NET\ADOMD.NET\160'
if adomd_path not in path:
    path.append(adomd_path)

# for x in path:
#     print(x, sep='\n')

from pyadomd import Pyadomd

json_path = "Sales & Returns Sample v201912.json"

with open(json_path, "r", encoding="utf-8-sig") as f:
    data = json.load(f)

# Flatten metrics into top-level columns
def flatten_event(event):
    flat = event.copy()
    metrics = flat.pop("metrics", {})
    flat.update(metrics)
    return flat

flat_events = [flatten_event(e) for e in data.get("events", [])]


# Load the 'events' list into a DataFrame
#df = pd.DataFrame(data.get("events", []))

df = pd.DataFrame(flat_events)

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', 20)    
#print(df.head())

filtered_df = df[df['name'] == 'Execute DAX Query']

# filtered_df = df[
#     (df['name'] == 'Execute DAX Query') &
#     (df['id'] == 'baa3553b-c0fa-4f0e-87d6-98ea260fecba')
# ]
filtered_df


# Connection string to your Power BI semantic model (update as needed)
connection_string = "Provider=MSOLAP;Data Source=localhost:61954;Initial Catalog=31499e00-1fda-40a9-b59e-a2227c83ab3d"

results_list = []

# Loop through the DataFrame and execute each DAX query
with Pyadomd(connection_string) as conn:
    #Loop over each query from the dataframe
    for idx, row in filtered_df.iterrows():
        query_id = row['id']
        query_text = row['QueryText']
        print(f"QueryText:\n{query_text}\n")
        print(f"Executing Query ID: {query_id}")

        try:
            with conn.cursor().execute(query_text) as cur:
                a = cur.has_rows
                results = cur.fetchall()
                b = cur.has_rows
                print(f"Has rows: {cur.has_rows}\n")
                print(f"Row count: {cur.rowcount}\n")
                print(f"Results for {query_id}: {results}\n")

                r2 = cur.nextresult()
                if (r2):
                    results = cur.fetchall()
                    c = cur.has_rows
                    print(f"Results for {query_id}: {results}\n")

        except Exception as e:
            print(f"Error executing query {query_id}: {e}\n")



QueryText:
DEFINE VAR __DS0FilterTable = 
	TREATAS({"June"}, 'LocalDateTable_d9fbe243-4814-4038-8eec-593e864a563b'[Month])

EVALUATE
	SUMMARIZECOLUMNS(
		__DS0FilterTable,
		"MinEmpty", IGNORE(CALCULATE(MIN('Tooltip Info'[Empty])))
	)

Executing Query ID: 82bd947a-c0cb-48f5-87aa-2db65515b25e
Has rows: True

Row count: 1

Results for 82bd947a-c0cb-48f5-87aa-2db65515b25e: [(' ',)]

QueryText:
DEFINE
	VAR __DS0FilterTable = 
		TREATAS({"June"}, 'LocalDateTable_d9fbe243-4814-4038-8eec-593e864a563b'[Month])

	VAR __DM3FilterTable = 
		FILTER(
			KEEPFILTERS(VALUES('Details'[Design Factor])),
			NOT('Details'[Design Factor] IN {"UI/UX"})
		)

	VAR __DS0Core = 
		FILTER(
			KEEPFILTERS(
				SUMMARIZECOLUMNS(
					ROLLUPADDISSUBTOTAL(
						ROLLUPGROUP('Details'[Design Factor], 'Details'[DFSort]), "IsGrandTotalRowTotal",
						ROLLUPGROUP('Details'[Topic], 'Details'[TSort]),
						"IsDM1Total",
						NONVISUAL(__DM3FilterTable)
					),
					__DS0FilterTable
				)
			),
			OR(
				OR(
					OR(
